## Objective
The challenge is to design and implement a working, open-source algorithm that can predict in-hospital mortality from routinely collected data at the time of presentation to a health facility in Uganda.

In [1]:
# Libraries
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import argparse
import time
import psutil
import json
import random
from math import sqrt

%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

from scipy import sparse
from scipy.stats import uniform, randint

import sklearn
from sklearn import preprocessing
from sklearn import model_selection
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, confusion_matrix
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.metrics import roc_curve, auc, recall_score, precision_score, f1_score, confusion_matrix, recall_score, roc_auc_score, average_precision_score, brier_score_loss
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict,TimeSeriesSplit, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.naive_bayes import MultinomialNB



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of category-encoders to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of pmdarima to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.1/486.1 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.8/21.8 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.1 M

In [2]:
# Load Data
data_path = "https://raw.githubusercontent.com/Kamaleswaran-Lab/The-2024-Pediatric-Sepsis-Challenge/refs/heads/main/SyntheticData_Training.csv"  # update this if needed
df = pd.read_csv(data_path, index_col=0)


## Data Exploration

In [3]:
df.head() # snapshot of the data

,agecalc_adm,height_cm_adm,weight_kg_adm,muac_mm_adm,hr_bpm_adm,rr_brpm_app_adm,sysbp_mmhg_adm,diasbp_mmhg_adm,temp_c_adm,spo2site1_pc_oxi_adm,...,cookloc_adm,lightfuel_adm,tobacco_adm,bednet_adm,hctpretransfusion_adm,hivstatus_adm,malariastatuspos_adm,lengthadm,caregiver_adm_new,inhospital_mortality
studyid_adm,,,,,,,,,,,,,,,,,,,,,
1,16.8,79.8,11.6,150.0,130.0,23.0,92.0,54.0,36.3,99.0,...,In a separate building/building space used as ...,Electric bulbs (national grid),Never,Always,Yes,HIV negative,No,2,Mother,0
2,46.1,93.0,13.6,151.0,115.0,35.0,72.0,42.0,36.8,92.0,...,In a separate building/building space used as ...,Tadooba,Never,Always,Yes,HIV negative,Yes,5,Mother,0
3,7.9,68.2,8.2,148.0,150.0,56.0,94.0,51.0,37.2,99.0,...,In a separate building/building space used as ...,Tadooba,Never,Always,Yes,HIV negative,No,4,Mother,0
4,38.2,95.0,12.0,138.0,134.0,38.0,94.0,57.0,37.6,98.0,...,In the house where you sleep,Electric bulbs (national grid),Never,Always,NaN,HIV negative,Yes,5,Mother,0
5,16.1,83.0,12.0,165.0,163.0,61.0,107.0,73.0,38.7,90.0,...,Outdoors in the open,Electric bulbs (national grid),Never,Sometimes,Yes,HIV negative,Yes,7,Mother,0


In [4]:
df.describe()

,agecalc_adm,height_cm_adm,weight_kg_adm,muac_mm_adm,hr_bpm_adm,rr_brpm_app_adm,sysbp_mmhg_adm,diasbp_mmhg_adm,temp_c_adm,spo2site1_pc_oxi_adm,...,alivechildren_adm,deadchildren_adm,hematocrit_gpdl_adm,lactate_mmolpl_adm,lactate2_mmolpl_adm,glucose_mmolpl_adm,sqi1_perc_oxi_adm,sqi2_perc_oxi_adm,lengthadm,inhospital_mortality
count,2686.00000,2672.000000,2677.000000,2673.000000,2685.000000,2681.000000,2684.000000,2685.000000,2685.000000,2682.000000,...,2682.00000,2685.000000,2229.000000,2222.000000,469.000000,2677.000000,2556.000000,2479.000000,2686.000000,2686.000000
mean,21.12338,79.659401,9.577265,138.076693,141.219367,46.179784,96.941133,54.594041,37.470503,94.750186,...,3.02349,0.354935,32.954240,2.774887,2.951599,6.320882,88.780516,78.004841,5.377141,0.044304
std,13.05725,11.110075,2.796121,15.934621,25.550612,14.871732,11.876974,11.005274,1.007606,6.389585,...,1.81805,0.810868,9.750002,2.123403,2.234946,2.512912,16.714758,23.383975,7.654527,0.205808
min,5.90000,56.000000,2.610000,75.000000,42.000000,17.000000,58.000000,15.000000,33.600000,57.000000,...,0.00000,0.000000,6.000000,0.500000,0.200000,0.000000,0.000000,0.000000,-85.000000,0.000000
25%,10.52500,71.500000,7.640000,130.000000,128.000000,35.000000,89.000000,48.000000,36.700000,93.000000,...,2.00000,0.000000,28.000000,1.500000,1.400000,5.000000,87.000000,68.000000,3.000000,0.000000
50%,16.90000,78.000000,9.000000,140.000000,143.000000,44.000000,96.000000,54.000000,37.200000,97.000000,...,3.00000,0.000000,34.000000,2.100000,2.300000,6.000000,96.000000,87.000000,4.000000,0.000000
75%,28.67500,86.700000,11.000000,149.000000,158.000000,56.000000,104.000000,62.000000,38.200000,99.000000,...,4.00000,0.000000,40.000000,3.300000,3.700000,7.200000,99.000000,96.000000,6.000000,0.000000
max,61.80000,122.000000,22.000000,190.000000,214.000000,116.000000,159.000000,103.000000,40.600000,100.000000,...,11.00000,9.000000,82.000000,18.900000,15.900000,24.200000,99.000000,99.000000,210.000000,1.000000


In [5]:
df.columns.tolist()

['agecalc_adm',
 'height_cm_adm',
 'weight_kg_adm',
 'muac_mm_adm',
 'hr_bpm_adm',
 'rr_brpm_app_adm',
 'sysbp_mmhg_adm',
 'diasbp_mmhg_adm',
 'temp_c_adm',
 'spo2site1_pc_oxi_adm',
 'spo2site2_pc_oxi_adm',
 'spo2other_adm',
 'momage_adm',
 'momagefirstpreg_adm',
 'householdsize_adm',
 'alivechildren_adm',
 'deadchildren_adm',
 'hematocrit_gpdl_adm',
 'lactate_mmolpl_adm',
 'lactate2_mmolpl_adm',
 'glucose_mmolpl_adm',
 'sqi1_perc_oxi_adm',
 'sqi2_perc_oxi_adm',
 'sex_adm',
 'spo2onoxy_adm',
 'oxygenavail_adm',
 'respdistress_adm',
 'caprefill_adm',
 'bcseye_adm',
 'bcsmotor_adm',
 'bcsverbal_adm',
 'admitabx_adm___1',
 'admitabx_adm___2',
 'admitabx_adm___3',
 'admitabx_adm___4',
 'admitabx_adm___5',
 'admitabx_adm___6',
 'admitabx_adm___7',
 'admitabx_adm___8',
 'admitabx_adm___9',
 'admitabx_adm___10',
 'admitabx_adm___11',
 'admitabx_adm___12',
 'admitabx_adm___13',
 'admitabx_adm___14',
 'admitabx_adm___15',
 'admitabx_adm___16',
 'admitabx_adm___17',
 'admitabx_adm___18',
 'admit

In [6]:
df.shape

(2686, 137)

In [7]:
numerical_features = [
        'agecalc_adm', 'height_cm_adm', 'weight_kg_adm', 'muac_mm_adm', 'hr_bpm_adm',
        'rr_brpm_app_adm', 'sysbp_mmhg_adm', 'diasbp_mmhg_adm', 'temp_c_adm',
        'spo2site1_pc_oxi_adm', 'spo2site2_pc_oxi_adm', 'spo2other_adm', 'momage_adm',
        'momagefirstpreg_adm', 'householdsize_adm', 'alivechildren_adm', 'deadchildren_adm',
        'hematocrit_gpdl_adm', 'lactate_mmolpl_adm', 'lactate2_mmolpl_adm',
        'glucose_mmolpl_adm', 'sqi1_perc_oxi_adm', 'sqi2_perc_oxi_adm'
    ]

In [8]:
categorical_features = [
        'sex_adm', 'spo2onoxy_adm', 'oxygenavail_adm', 'respdistress_adm', 'caprefill_adm',
        'bcseye_adm', 'bcsmotor_adm', 'bcsverbal_adm', 'bcgscar_adm', 'vaccmeasles_adm',
        'vaccmeaslessource_adm', 'vaccpneumoc_adm', 'vaccpneumocsource_adm', 'vaccdpt_adm',
        'vaccdptsource_adm', 'priorweekabx_adm', 'priorweekantimal_adm',
        'symptoms_adm___1', 'symptoms_adm___2', 'symptoms_adm___3', 'symptoms_adm___4',
        'symptoms_adm___5', 'symptoms_adm___6', 'symptoms_adm___7', 'symptoms_adm___8',
        'symptoms_adm___9', 'symptoms_adm___10', 'symptoms_adm___11', 'symptoms_adm___12',
        'symptoms_adm___13', 'symptoms_adm___14', 'symptoms_adm___15', 'symptoms_adm___16',
        'symptoms_adm___18',
        'comorbidity_adm___1', 'comorbidity_adm___2', 'comorbidity_adm___3', 'comorbidity_adm___4',
        'comorbidity_adm___5', 'comorbidity_adm___6', 'comorbidity_adm___7', 'comorbidity_adm___8',
        'comorbidity_adm___9', 'comorbidity_adm___10', 'comorbidity_adm___11', 'comorbidity_adm___12',
        'priorhosp_adm', 'prioryearwheeze_adm', 'prioryearcough_adm', 'diarrheaoften_adm',
        'tbcontact_adm', 'feedingstatus_adm', 'exclbreastfed_adm', 'nonexclbreastfed_adm',
        'totalbreastfed_adm', 'deliveryloc_adm', 'birthattend_adm', 'duedateknown_adm',
        'birthdetail_adm___1', 'birthdetail_adm___2', 'birthdetail_adm___3',
        'birthdetail_adm___4', 'birthdetail_adm___5', 'birthdetail_adm___6',
        'travelmethod_adm', 'traveldist_adm', 'badhealthduration_adm', 'caregiver_adm_new',
        'caregiverage_adm', 'caregivermarried_adm', 'momalive_adm', 'momageknown_adm',
        'momagefirstpregknown_adm', 'momedu_adm', 'momhiv_adm', 'watersource_adm', 'waterpure_adm',
        'cookfuel_adm___1', 'cookfuel_adm___2', 'cookfuel_adm___3', 'cookfuel_adm___4',
        'cookfuel_adm___5', 'cookfuel_adm___6', 'cookfuel_adm___7',
        'cookloc_adm', 'lightfuel_adm', 'tobacco_adm', 'bednet_adm',
        'hctpretransfusion_adm', 'hivstatus_adm', 'malariastatuspos_adm'
    ]

In [9]:
 ## Function to calculate Net Benefit
def calculate_net_benefit(y_true, y_pred, threshold):
    """
    Calculate the net benefit of predictions.

    Parameters:
    - y_true (array-like): Ground truth binary labels (0 or 1).
    - y_pred (array-like): Predicted probabilities.
    - threshold (float): Threshold probability for classification (default is 0.5).

    Returns:
    - float: Net benefit score.
    """
    tp = np.sum((y_true == 1) & (y_pred >= threshold))
    fp = np.sum((y_true == 0) & (y_pred >= threshold))
    n = len(y_true)

    # Calculate Net Benefit
    net_benefit = (tp / n) - ((threshold / (1 - threshold)) * (fp / n))
    return net_benefit




In [10]:
# Function to calculate ECE (Estimated Calibration Error)
def calculate_ece(probs, labels, n_bins=10):
  bin_edges = np.linspace(0, 1, n_bins + 1)
  ece = 0
  for i in range(n_bins):
    bin_mask = (probs > bin_edges[i]) & (probs <= bin_edges[i + 1])
    bin_size = np.sum(bin_mask)
    if bin_size > 0:
      bin_acc = np.mean(labels[bin_mask])
      bin_conf = np.mean(probs[bin_mask])
      ece += bin_size * np.abs(bin_acc - bin_conf) / len(probs)
  return ece

In [11]:
# Define features (X) and target (y)
X = df[numerical_features + categorical_features]
y = df['inhospital_mortality']

# Create preprocessing pipelines for numerical and categorical features
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Create a column transformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough' # Keep other columns if any
)

# Model definitions (as provided by the user, with soft voting for ensemble)
model1=LogisticRegression(random_state=22,C=0.000000001,solver='liblinear',max_iter=200)
model2=GaussianNB()
model3=RandomForestClassifier(n_estimators=200,random_state=22)
model4=GradientBoostingClassifier(n_estimators=200)
model5=KNeighborsClassifier()
model6=DecisionTreeClassifier()
model7=LinearDiscriminantAnalysis()
model8=BaggingClassifier()
Ensembled_model=VotingClassifier(estimators=[('lr', model1), ('gn', model2), ('rf', model3),('gb',model4),('kn',model5),('dt',model6),('lda',model7), ('bc',model8)], voting='soft')

# List to store all evaluation results
all_results = []

# Evaluate each model
print("Starting model evaluation...")
for model, label in zip([model1, model2, model3, model4, model5, model6, model7, model8, Ensembled_model],
                        ['Logistic Regression', 'Naive Bayes', 'Random Forest', 'Gradient Boosting', 'KNN',
                         'Decision Tree', 'LDA', 'Bagging Classifier', 'Ensemble']):
    print(f"Evaluating {label}...")
    # Create a pipeline that includes preprocessing and the model
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])

    # Stratified K-Fold for consistent and properly sampled cross-validation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Calculate Accuracy using cross_val_score
    accuracy_scores = cross_val_score(pipeline, X, y, cv=skf, scoring='accuracy')
    accuracy_mean = accuracy_scores.mean()

    # Get predicted probabilities for ECE and Net Benefit
    try:
        y_pred_proba_cv = cross_val_predict(pipeline, X, y, cv=skf, method='predict_proba')[:, 1]
    except AttributeError:
        # Fallback to predict if predict_proba is not available
        print(f"Warning: Model {label} does not support predict_proba. Using predict for ECE/Net Benefit (less accurate).")
        y_pred_class_cv = cross_val_predict(pipeline, X, y, cv=skf, method='predict')
        # Convert binary predictions to pseudo-probabilities for ECE/Net Benefit calculation
        y_pred_proba_cv = y_pred_class_cv.astype(float)

    # Ensure y_true is aligned with cross_val_predict output
    y_true_cv = y.values

    # Calculate ECE
    ece_score = calculate_ece(y_pred_proba_cv, y_true_cv)

    # Calculate Net Benefit (using a default threshold of 0.5)
    net_benefit_score = calculate_net_benefit(y_true_cv, y_pred_proba_cv, threshold=0.5)

    all_results.append({
        'Model': label,
        'Accuracy': accuracy_mean,
        'ECE': ece_score,
        'Net Benefit (threshold=0.5)': net_benefit_score
    })

    print(f"Model: {label}, Accuracy: {accuracy_mean:.4f}, ECE: {ece_score:.4f}, Net Benefit: {net_benefit_score:.4f}\n")

# Display results in a DataFrame
results_df = pd.DataFrame(all_results)
display(results_df)


Starting model evaluation...
Evaluating Logistic Regression...
Model: Logistic Regression, Accuracy: 0.9557, ECE: 0.4557, Net Benefit: 0.0000

Evaluating Naive Bayes...
Model: Naive Bayes, Accuracy: 0.1724, ECE: 0.8209, Net Benefit: -0.7833

Evaluating Random Forest...
Model: Random Forest, Accuracy: 0.9557, ECE: 0.0082, Net Benefit: 0.0007

Evaluating Gradient Boosting...
Model: Gradient Boosting, Accuracy: 0.9512, ECE: 0.0270, Net Benefit: -0.0048

Evaluating KNN...
Model: KNN, Accuracy: 0.9564, ECE: 0.0077, Net Benefit: 0.0007

Evaluating Decision Tree...
Model: Decision Tree, Accuracy: 0.9188, ECE: 0.0484, Net Benefit: -0.0395

Evaluating LDA...
Model: LDA, Accuracy: 0.9401, ECE: 0.0473, Net Benefit: -0.0156

Evaluating Bagging Classifier...
Model: Bagging Classifier, Accuracy: 0.9553, ECE: 0.0225, Net Benefit: -0.0089

Evaluating Ensemble...
Model: Ensemble, Accuracy: 0.9516, ECE: 0.1577, Net Benefit: -0.0026



,Model,Accuracy,ECE,Net Benefit (threshold=0.5)
0,Logistic Regression,0.955696,0.455679,0.000000
1,Naive Bayes,0.172363,0.820923,-0.783321
2,Random Forest,0.955696,0.008153,0.000745
3,Gradient Boosting,0.951229,0.027033,-0.004840
4,KNN,0.956441,0.007669,0.000745
5,Decision Tree,0.918843,0.048399,-0.039464
6,LDA,0.940064,0.047320,-0.015637
7,Bagging Classifier,0.955326,0.022487,-0.008935
8,Ensemble,0.951602,0.157739,-0.002606
